# GPU এবং Hardware Fundamentals

এই notebook `example.py`-এর দুটি অংশ চালায়:

**Part A** — একটি বাস্তব, গণনাযোগ্য arithmetic-intensity ক্যালকুলেটর (README section 5), যা Transformer-আকৃতির matmuls-এ প্রয়োগ করে দেখায়—বাস্তব সংখ্যা দিয়ে—কেন PREFILL compute-bound আর DECODE memory-bound (README section 6), একটি সৎ-আনুমানিক বাস্তব GPU-র ridge point-এর সাপেক্ষে। পাশাপাশি batch size জুড়ে decode-এর arithmetic intensity-ও ঘোরানো হয়, যাতে কংক্রিটভাবে দেখা যায় কতগুলো সমকালীন decode request লাগে ridge point-কে আবার অতিক্রম করতে (continuous batching-এর বাস্তব যুক্তি)।

**Part B** — একই গুণগত প্রভাবের একটি CPU-পরিমাপনযোগ্য ANALOGY (স্পষ্টভাবে analogy হিসেবে লেবেলযুক্ত, GPU reproduction নয়): একটি বড় batched matmul বনাম অনেকগুলো পৃথক single-row matmul, এই স্ক্রিপ্ট যে CPU-তে চলে সেখানে বাস্তবে সময় মেপে। এই মেশিনে HBM/SRAM traffic সরাসরি মাপার মতো আলাদা GPU নেই — CPU cache levels (L1/L2/L3) GPU-র SRAM/HBM বিভক্তির সাথে মোটামুটি সাদৃশ্যপূর্ণ ভূমিকা পালন করে, অনেক ছোট স্কেলে এবং ভিন্ন সংখ্যা দিয়ে, কিন্তু একই QUALITATIVE আকৃতি সহ।

**চালানোর নিয়ম:** উপরের দিক থেকে নিচের দিকে প্রতিটি cell চালান। নোটবুকটি top-to-bottom চলে; প্রতিটি সেকশনের demo функции তার নিজ নিজ cell-এর শেষেই চালানো হয়।

In [ ]:
import time
import torch

torch.manual_seed(0)

## Part A: Arithmetic intensity — prefill বনাম decode

বাস্তব Transformer-আকৃতির matmuls-এর উপর arithmetic intensity গণনা করা হয়, README section 5-এর ফর্মুলা অনুযায়ী। তারপর section 6-এর শ্রেণীবিন্যাস — compute-bound বনাম memory-bound — একটি আধুনিক data-center GPU-র আনুমানিক ridge point-এর বিরুদ্ধে প্রয়োগ করা হয়। শেষ অংশে batch size বাড়িয়ে দেখা হয়, কতগুলো সমকালীন decode request-এ decode আবার ridge point-এর উপরে চলে যায়।

In [ ]:
# ---------------------------------------------------------------------------
# PART A: ARITHMETIC INTENSITY -- PREFILL বনাম DECODE-এর জন্য বাস্তবভাবে গণনা করা
# ---------------------------------------------------------------------------

BYTES_PER_ELEM = 2  # bf16/fp16 -- standard training/inference dtype (Phase 04 Lesson 4)

# একটি আধুনিক data-center GPU-র bf16 tensor cores-এর জন্য একটি সৎ-আনুমানিক,
# স্পষ্টভাবে লেবেলযুক্ত রেফারেন্স ridge point: peak FLOPs / peak HBM bandwidth।
# বাস্তব chips ভিন্ন হয়; এটি একটি গোলাকার, order-of-magnitude stand-in (মোটামুটি
# যা একটি NVIDIA A100-এ দাঁড়ায়: ~312 TFLOPs bf16 / ~2 TB/s HBM bandwidth), কোনো
# নির্দিষ্ট product-এর জন্য নির্ভুল স্পেসিফিকেশন নয়।
PEAK_TFLOPS = 300.0          # approx., bf16 tensor-core peak, TFLOPs/s
PEAK_BANDWIDTH_TB_S = 2.0    # approx., HBM bandwidth, TB/s
RIDGE_POINT = (PEAK_TFLOPS * 1e12) / (PEAK_BANDWIDTH_TB_S * 1e12)  # FLOPs/byte


def matmul_flops(m, k, n):
    """একটি (m, k) @ (k, n) matmul-এর FLOPs (README section 4)।"""
    return 2 * m * k * n


def matmul_bytes(m, k, n, bytes_per_elem=BYTES_PER_ELEM):
    """একটি (m, k) @ (k, n) matmul-এর জন্য সরানো bytes: activations পড়তে হবে,
    weights পড়তে হবে, output লিখতে হবে। এটি একটি naive DRAM-traffic model (ধরে
    নেয় প্রতিটি matrix HBM-এ/থেকে একবার পূর্ণ যাত্রা করে) -- roofline analysis
    ব্যবহার করে এমন মোটামুটি হিসাব, এবং একটি non-tiled, unfused kernel আসলে যা
    করে। একটি tiled kernel (যেমন Phase 02 Lesson 7-এর FlashAttention) SOME
    অপারেশনের জন্য এর চেয়ে ভালো করতে পারে; বেশিরভাগ inference stack-এর সাধারণ
    dense matmuls করে না।"""
    activations = m * k
    weights = k * n
    output = m * n
    return (activations + weights + output) * bytes_per_elem


def arithmetic_intensity(m, k, n, bytes_per_elem=BYTES_PER_ELEM):
    flops = matmul_flops(m, k, n)
    bts = matmul_bytes(m, k, n, bytes_per_elem)
    return flops / bts, flops, bts


def classify(ai):
    return "COMPUTE-BOUND" if ai > RIDGE_POINT else "MEMORY-BOUND"


def prefill_vs_decode_demo():
    print("=" * 90)
    print("PART A.1: ARITHMETIC INTENSITY -- PREFILL vs. DECODE, ON A REAL FFN-SHAPED MATMUL")
    print("=" * 90)
    d_model = 4096
    d_ff = 4 * d_model   # the FFN up-projection shape (Phase 02 Lesson 5 section 5)
    prefill_seq_len = 2048

    print(f"Weight matrix: ({d_model}, {d_ff})  [one FFN up-projection, bf16]")
    print(f"Reference ridge point: {PEAK_TFLOPS:.0f} TFLOPs / {PEAK_BANDWIDTH_TB_S:.1f} TB/s "
          f"= {RIDGE_POINT:.1f} FLOPs/byte (approx., a modern data-center GPU)\n")

    ai_prefill, flops_prefill, bytes_prefill = arithmetic_intensity(prefill_seq_len, d_model, d_ff)
    ai_decode, flops_decode, bytes_decode = arithmetic_intensity(1, d_model, d_ff)

    header = f"{'phase':10}{'m (tokens)':>12}{'FLOPs':>16}{'bytes moved':>16}{'AI (FLOPs/B)':>16}{'classification':>18}"
    print(header)
    print("-" * len(header))
    print(f"{'prefill':10}{prefill_seq_len:>12,}{flops_prefill:>16,.3g}{bytes_prefill:>16,}"
          f"{ai_prefill:>16,.1f}{classify(ai_prefill):>18}")
    print(f"{'decode':10}{1:>12,}{flops_decode:>16,.3g}{bytes_decode:>16,}"
          f"{ai_decode:>16,.3f}{classify(ai_decode):>18}")

    print(f"\n-> Prefill's arithmetic intensity ({ai_prefill:,.0f} FLOPs/byte) sits ~"
          f"{ai_prefill / RIDGE_POINT:.0f}x ABOVE the ridge point -- comfortably compute-bound.")
    print(f"   Decode's ({ai_decode:.3f} FLOPs/byte) sits ~{RIDGE_POINT / ai_decode:.0f}x BELOW it --")
    print(f"   deeply memory-bound. Note decode's AI lands almost exactly at "
          f"2/{BYTES_PER_ELEM} = {2 / BYTES_PER_ELEM:.2f}:")
    print(f"   with only ONE token's worth of activations, the weight matrix (which dominates the")
    print(f"   byte count) contributes almost exactly one multiply-add (2 FLOPs) per")
    print(f"   {BYTES_PER_ELEM}-byte element it holds -- there is structurally no way for a single")
    print(f"   token's decode step to do more arithmetic per byte than that, no matter the model size.")


def decode_batching_sweep_demo():
    print("\n" + "=" * 90)
    print("PART A.2: HOW MANY CONCURRENT DECODE REQUESTS TO CROSS BACK OVER THE RIDGE POINT?")
    print("=" * 90)
    d_model = 4096
    d_ff = 4 * d_model
    print("Same FFN weight matrix, but now m = number of CONCURRENT decode requests batched")
    print("together in one matmul (each contributing exactly one token) -- exactly what")
    print("continuous batching (Lesson 4) does.\n")

    batch_sizes = [1, 8, 32, 64, 128, 192, 256, 512]
    header = f"{'batch size (m)':>16}{'AI (FLOPs/byte)':>20}{'classification':>18}"
    print(header)
    print("-" * len(header))
    crossed = None
    for b in batch_sizes:
        ai, _, _ = arithmetic_intensity(b, d_model, d_ff)
        label = classify(ai)
        print(f"{b:>16}{ai:>20,.1f}{label:>18}")
        if crossed is None and ai > RIDGE_POINT:
            crossed = b

    # crossover batch size-কে আনুমানিকভাবে binary-search করে একটি কংক্রিট সংখ্যা বের করি।
    lo, hi = 1, 4096
    while hi - lo > 1:
        mid = (lo + hi) // 2
        ai_mid, _, _ = arithmetic_intensity(mid, d_model, d_ff)
        if ai_mid > RIDGE_POINT:
            hi = mid
        else:
            lo = mid
    print(f"\n-> Crosses from memory-bound to compute-bound somewhere around batch size ~{hi} "
          f"concurrent decode requests")
    print(f"   (at this FFN shape and reference ridge point). This is exactly why real LLM serving")
    print(f"   systems need surprisingly LARGE numbers of concurrent requests batched together")
    print(f"   before decode stops being memory-bound -- and exactly the gap continuous batching")
    print(f"   (Lesson 4: Serving Frameworks) is built to keep filled with real, waiting traffic.")


prefill_vs_decode_demo()
decode_batching_sweep_demo()

## Part B: একটি CPU-পরিমাপনযোগ্য analogy

একটি batched matmul বনাম একই মোট কাজ আলাদা আলাদা single-row matmul-এ — CPU cache hierarchy প্রশিক্ষণের analogy হিসেবে। এখানে যা মাপা হয় তা GPU-র HBM/SRAM ব্যবধানের প্রতিরূপ, একই গুণগত আকৃতিসহ।

In [ ]:
# ---------------------------------------------------------------------------
# PART B: একটি CPU-পরিমাপনযোগ্য ANALOGY (স্পষ্টতই GPU reproduction নয়)
# ---------------------------------------------------------------------------

def cpu_batching_analogy_demo():
    print("\n" + "=" * 90)
    print("PART B: A CPU-MEASURABLE ANALOGY -- BATCHED vs. ONE-ROW-AT-A-TIME MATMULS")
    print("=" * 90)
    print("HONEST CAVEAT: this machine has no discrete GPU to measure HBM/SRAM traffic on.")
    print("What follows is an ANALOGY, not a reproduction: a CPU's cache hierarchy (L1/L2/L3)")
    print("plays a loosely similar role to a GPU's SRAM-vs-HBM split -- much smaller gap, very")
    print("different absolute numbers -- but the same QUALITATIVE effect shows up: reusing a")
    print("weight matrix across many rows of work in one call is far more efficient per row than")
    print("paying to touch that same weight matrix once per row, one row at a time.\n")

    d = 2048
    n_tokens = 4096
    weight = torch.randn(d, d)

    # "Prefill-shaped": ONE matmul call-এ সব n_tokens row-কে weight-এর বিরুদ্ধে গুণ করা হয়।
    x_batched = torch.randn(n_tokens, d)
    torch.mm(x_batched[:8], weight)  # warm-up, প্রথম-কল overhead যেন timing-কে skew না করে
    start = time.perf_counter()
    out_batched = torch.mm(x_batched, weight)
    batched_time = time.perf_counter() - start

    # "Decode-shaped": মোট row-এর SAME সংখ্যা, কিন্তু একবারে ONE row, n_tokens আলাদা কল।
    x_rows = [torch.randn(1, d) for _ in range(n_tokens)]
    torch.mm(x_rows[0], weight)  # warm-up
    start = time.perf_counter()
    for row in x_rows:
        _ = torch.mm(row, weight)
    sequential_time = time.perf_counter() - start

    batched_tokens_per_sec = n_tokens / batched_time
    sequential_tokens_per_sec = n_tokens / sequential_time

    print(f"Weight matrix: ({d}, {d}), fp32. {n_tokens} total rows of work, either way.\n")
    header = f"{'mode':28}{'wall-clock':>14}{'tokens/sec':>16}"
    print(header)
    print("-" * len(header))
    print(f"{'one batched matmul':28}{batched_time * 1000:>11.2f} ms{batched_tokens_per_sec:>16,.0f}")
    print(f"{'N separate 1-row matmuls':28}{sequential_time * 1000:>11.2f} ms{sequential_tokens_per_sec:>16,.0f}")

    speedup = sequential_time / batched_time
    print(f"\n-> The single batched call reached {speedup:.1f}x the tokens/sec of doing the exact")
    print(f"   same total arithmetic one row at a time. Two effects are bundled together here")
    print(f"   (both real, on this CPU, right now): better reuse of the weight matrix while it's")
    print(f"   resident in a fast cache level, and less per-call dispatch overhead. A real GPU's")
    print(f"   HBM/SRAM gap and warp-level parallelism (README sections 1-2) produce the same")
    print(f"   qualitative shape far more dramatically -- this is the same shape, at a much")
    print(f"   smaller scale, on completely different hardware.")


cpu_batching_analogy_demo()

In [ ]:
def main():
    prefill_vs_decode_demo()
    decode_batching_sweep_demo()
    cpu_batching_analogy_demo()


main()